In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/test.csv")

In [6]:
df["LotFrontage"] = df["LotFrontage"].fillna(
    df.groupby("Neighborhood")["LotFrontage"].transform("median"))

df["Alley"] = df["Alley"].fillna("NA")

df.loc[df["MasVnrArea"]==0,"MasVnrType"] = "None"

df.loc[(df["MasVnrArea"].isna() & df["MasVnrType"].isna()),"MasVnrType"] = "None"
df.loc[(df["MasVnrArea"].isna() & df["MasVnrType"].isna()),"MasVnrArea"] = 0

df.loc[(df["MasVnrType"] == "None") & (df["MasVnrArea"].isna()), "MasVnrArea"] = 0

df.loc[
    (df["MasVnrArea"] > 0) & (df["MasVnrType"].isna()),
    "MasVnrType"
] = df["MasVnrType"].mode()[0]

df["BsmtQual"] = df["BsmtQual"].fillna("NA")
df["BsmtCond"] = df["BsmtCond"].fillna("NA") 

df.loc[
    (df["BsmtCond"] == "NA") &
    (df["BsmtQual"] == "NA") &
    (df["BsmtExposure"].isna()),
    "BsmtExposure"
] = "NA"

df["BsmtExposure"] = df["BsmtExposure"].fillna("No") 

df["BsmtFinType1"] = df["BsmtFinType1"].fillna(0) 

df.loc[(df["BsmtQual"] != "NA") & (df["BsmtCond"] != "NA") & (df["BsmtExposure"] != "NA") &
   (df["BsmtFinType2"].isna()),"BsmtFinType2"] = "Unf"

df["BsmtFinType2"] = df["BsmtFinType2"].fillna("NA") 

df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

df.loc[(df["Fireplaces"] == 0),"FireplaceQu"] = "NA"

df["GarageType"] = df["GarageType"].fillna("NA")
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["GarageYrBlt"].mode()[0])
df["GarageFinish"] = df["GarageFinish"].fillna("NA")
df["GarageQual"] = df["GarageQual"].fillna("NA")
df["GarageCond"] = df["GarageCond"].fillna("NA")

df.loc[(df["PoolArea"] == 0),"PoolQC"] = "NA"

df["Fence"] = df["Fence"].fillna("NA")

df["MiscFeature"] = df["MiscFeature"].fillna("NA")

In [13]:
# --- Fix remaining NaNs (Ames test quirks) ---

# 1) CATEGORICAL: fill with mode (or known default)
cat_mode_cols = [
    "Utilities",
    "Exterior1st",
    "Exterior2nd",
    "KitchenQual",
    "SaleType",
]

for col in cat_mode_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

# Functional: Kaggle/AMES common default for missing is "Typ"
if "Functional" in df.columns:
    df["Functional"] = df["Functional"].fillna("Typ")

# PoolQC: missing typically means no pool
if "PoolQC" in df.columns:
    df["PoolQC"] = df["PoolQC"].fillna("NA")

# MSZoning: try neighborhood-wise mode; fallback to global mode
if "MSZoning" in df.columns:
    if "Neighborhood" in df.columns:
        df["MSZoning"] = df["MSZoning"].fillna(
            df.groupby("Neighborhood")["MSZoning"].transform(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
        )
    df["MSZoning"] = df["MSZoning"].fillna(df["MSZoning"].mode()[0])

# 2) NUMERIC (basement/garage fields in test): fill with 0
num_zero_cols = [
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "GarageCars",
    "GarageArea",
]

for col in num_zero_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# --- sanity check ---
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(nan_cols)
print("Total NaNs:", int(nan_counts.sum()))

Series([], dtype: int64)
Total NaNs: 0


In [14]:
df.columns[df.isna().any()]

Index([], dtype='object')

In [15]:
df.to_csv("data/test_cleaned.csv",index = False)